In [3]:
import nanopub
print(nanopub.__version__)

2.1.0


In [1]:
import pandas as pd
flora = pd.read_csv("https://raw.githubusercontent.com/forrtproject/fred-data/refs/heads/main/output/flora.csv")


# FLoRA filtering

In [ ]:
flora_filtered = flora[flora["outcome"].isin(["successful", "failed", "mixed"])]


outcome
successful    1019
failed         989
mixed          625
Name: count, dtype: int64

In [64]:
# multi-study papers: several flora rows share one paper identifier but each has its own url_r.
# For these, work = url_r and the paper (doi_r or alt_identifier_r) goes into `collection`.

def shared_id_unique_url(df, key):
    """Rows whose `key` is shared across several rows, where every row in that
    group has a url_r and no two rows in the group have the same url_r."""
    d = df[df[key].notna()]
    g = d.groupby(key)["url_r"]
    qualifies = (
        (g.transform("size") > 1)                          # key shared across rows
        & (g.transform(lambda s: s.notna().all()))         # every row has a url_r
        & (g.transform("nunique") == g.transform("size"))  # url_r all different
    )
    return d[qualifies]

# A: doi_r shared across rows, each with a unique url_r
group_a = shared_id_unique_url(flora, "doi_r")
# B: no doi_r, alt_identifier_r shared across rows, each with a unique url_r
group_b = shared_id_unique_url(flora[flora["doi_r"].isna()], "alt_identifier_r")

multi_studies = pd.concat([group_a.assign(match="doi_r"),
                           group_b.assign(match="alt_identifier_r")])
multi_study_ids = set(multi_studies["id"])

print(len(group_a), "rows via doi_r,", len(group_b), "rows via alt_identifier_r")
multi_studies.sort_values(["doi_r", "alt_identifier_r"])[["id", "match", "doi_r", "alt_identifier_r", "url_r"]]
multi_studies[multi_studies["url_r"] == "https://osf.io/bfs6p"][["id", "match", "doi_r", "alt_identifier_r", "url_r"]   ]

61 rows via doi_r, 179 rows via alt_identifier_r


,id,match,doi_r,alt_identifier_r,url_r
2337,FLORA-002635,alt_identifier_r,NaN,10.1038/s41586-025-10078-y,https://osf.io/bfs6p


# test df for testing

In [77]:
test = flora_filtered[
    (flora_filtered["doi_r"] == "10.1097/mlr.0000000000002256")
    | (flora_filtered["url_r"] == "https://osf.io/4rch2")
]
test[["id", "doi_r", "alt_identifier_r", "url_r", "outcome"]]
test.head()

,id,id_md5,doi_o,alt_identifier_o,doi_o_hash,title_o,author_o,journal_o,year_o,volume_o,...,abstract_r,outcome_computation,outcome_computational_quote,out_quote_computational_source,outcome_robustness,outcome_robustness_quote,out_quote_robust_source,flora_id,source_display_id,source_record_id
160,FLORA-000162,9195522ecff3f3dca603b9dd919e232d,10.1007/s11606-021-06676-x,NaN,631,Comparative Effectiveness of a Complex Care Pr...,"[{""given"":""Douglas W."",""family"":""Roblin"",""sequ...",Journal of General Internal Medicine,2021,36,...,Background/Objective:Evidence about the patien...,NaN,NaN,NaN,NaN,NaN,NaN,REPL-000164,REPL-000164,2e145b7b-ddf8-4e67-9d6f-07dda216ca91
2344,FLORA-002642,9f68f1ed066cf7b62a8ce11a56b213a9,10.1016/j.socscimed.2017.03.035,NaN,896,The more the heavier? Family size and childhoo...,"[{""given"":""Ashlesha"",""family"":""Datar"",""sequenc...",Social Science & Medicine,2017,180,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,REPL-000445,REPL-000445,7e3b768b-b656-4341-9d0f-8c504c9ffba3


# required fields

In [79]:
import requests
import pandas as pd
from rdflib import Dataset, Namespace, RDF, RDFS

TEMPLATES = [
    "https://w3id.org/np/RAUO3ykIwaMXNs8Do-pGujh1ELhmux_ajG6sAO_0Jk4EA",
    "https://w3id.org/np/RAZR5s1rLpW8GTHXorx6zZL1cIUPxSPjzD32lL4aDUh2k",
    "https://w3id.org/np/RAINLsq3Dj9_JhPfJqtmozdSCWufYB0NcOXOHd2BMhKnk",
    "https://w3id.org/np/RAv5GSYMLrwVmH7XUxWG05DNdV9BKV3et2Rxdf8LcD7lc"
]

NT = Namespace("https://w3id.org/np/o/ntemplate/")

def template_fields(template_uri):
    """Return a table of the fields (placeholders) in a nanopub template."""
    r = requests.get(template_uri, headers={"Accept": "application/trig"})
    r.raise_for_status()
    ds = Dataset()
    ds.parse(data=r.text, format="trig")
    g = next(c for c in ds.graphs() if str(c.identifier).endswith("/assertion"))
    tmpl = next(g.subjects(RDF.type, NT.AssertionTemplate))

    def walk(st, optional=False, repeatable=False):
        types = set(g.objects(st, RDF.type))
        optional |= NT.OptionalStatement in types
        repeatable |= NT.RepeatableStatement in types
        subs = list(g.objects(st, NT.hasStatement))
        if subs:  # grouped statement
            for s in subs:
                yield from walk(s, optional, repeatable)
        else:
            yield st, optional, repeatable

    fields = {}
    for st in g.objects(tmpl, NT.hasStatement):
        for leaf, opt, rep in walk(st):
            for pos in (RDF.subject, RDF.predicate, RDF.object):
                node = g.value(leaf, pos)
                ptypes = [t for t in g.objects(node, RDF.type)
                          if str(t).startswith(str(NT)) and "Placeholder" in str(t)]
                if not ptypes:
                    continue
                f = fields.setdefault(node, {
                    "field": str(node).rsplit("/", 1)[-1],
                    "label": str(g.value(node, RDFS.label) or ""),
                    "type": str(ptypes[0]).rsplit("/", 1)[-1],
                    "required": False,
                    "repeatable": False,
                })
                f["required"] |= not opt
                f["repeatable"] |= rep
    return pd.DataFrame(fields.values())

template_fields(TEMPLATES[3])



,field,label,type,required,repeatable
0,work,DOI (https://doi.org/10...) or other URL of th...,ExternalUriPlaceholder,True,False
1,cites,select the citation type,RestrictedChoicePlaceholder,True,False
2,cited,DOI (https://doi.org/10...) or other URL of th...,ExternalUriPlaceholder,True,False
3,outcome,Quote the replication/reproduction outcome,LongLiteralPlaceholder,True,False
4,collection,DOI (https://doi.org/10...) or other URL of th...,ExternalUriPlaceholder,False,False


# Nanopubs

In [ ]:
from rdflib import Graph, Literal, Namespace, URIRef, RDF, RDFS
from nanopub import Nanopub, NanopubConf, load_profile

TEMPLATE = URIRef("https://w3id.org/np/RAv5GSYMLrwVmH7XUxWG05DNdV9BKV3et2Rxdf8LcD7lc")
CITO  = Namespace("http://purl.org/spar/cito/")
FORRT = Namespace("https://w3id.org/sciencelive/o/terms/")
NT    = Namespace("https://w3id.org/np/o/ntemplate/")
DCT   = Namespace("http://purl.org/dc/terms/")
PROV = Namespace("http://www.w3.org/ns/prov#")
FLORA = URIRef("https://forrt.org/flora-replication-atlas/")
PROV_TEMPLATE = URIRef("https://w3id.org/np/RAGxHxC8z_PnSSrfMIBb5vWdhdlZAj_VZYyMt0C7ABOaM")



OUTCOME_TO_CITO = {
    "successful": CITO.confirms,
    "failed":     CITO.refutes,
    "mixed":      CITO.discusses,
}

def doi_uri(doi):
    """Turn flora identifiers into URIs: bare DOIs -> https://doi.org/..., other bare links -> https://..."""
    doi = str(doi).strip()
    if doi.startswith("http"):
        return URIRef(doi)
    if doi.startswith("10."):
        return URIRef("https://doi.org/" + doi)
    return URIRef("https://" + doi)

def require(row, *cols):
    for col in cols:
        if pd.isna(row[col]):
            raise ValueError(f"Required column {col!r} is empty")

def build_nanopub(row, conf, multi_ids=frozenset()):
    outcome = str(row["outcome"]).strip().lower()
    if outcome not in OUTCOME_TO_CITO:
        raise ValueError(f"Can't map outcome {row['outcome']!r} to a CiTO relation")
    require(row, "doi_o", "outcome_quote")

    if row["id"] in multi_ids:
        # multi-study paper: the individual study is the work, the paper is the collection
        require(row, "url_r")
        work = doi_uri(row["url_r"])
        paper = row["doi_r"] if pd.notna(row["doi_r"]) else row["alt_identifier_r"]
        collection = doi_uri(paper)
    else:
        require(row, "doi_r")
        work = doi_uri(row["doi_r"])
        collection = None
    cited = doi_uri(row["doi_o"])

    assertion = Graph()
    assertion.bind("cito", CITO)
    assertion.bind("forrt", FORRT)
    assertion.bind("dct", DCT)
    assertion.add((work, RDF.type, FORRT["FORRT-Replication-Study"]))               # st01
    assertion.add((work, OUTCOME_TO_CITO[outcome], cited))                          # st02
    assertion.add((work, FORRT["FORRT-Replication-Outcome"],
                   Literal(str(row["outcome_quote"]).strip())))                     # st03
    if collection is not None:                                                      # st04 (optional)
        assertion.add((work, DCT.isPartOf, collection))

    np = Nanopub(conf=conf, assertion=assertion)
   # provenance: follow the template's default provenance (attributed to creator + FLoRA)
    np.provenance.bind("prov", PROV)
    np.provenance.add((np.metadata.assertion, PROV.wasAttributedTo, FLORA))
    np.provenance.add((FLORA, RDFS.label, Literal("FLoRA")))

    this = np.metadata.namespace[""]
    np.pubinfo.bind("nt", NT)
    np.pubinfo.add((this, NT.wasCreatedFromTemplate, TEMPLATE))
    np.pubinfo.add((this, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
    np.pubinfo.add((this, RDFS.label, Literal(f"Replication outcome of: {work}")))
    return np

np_conf = NanopubConf(
    profile=load_profile(),
    use_test_server=True,
    add_prov_generated_time=True,
    attribute_assertion_to_profile=True,
    attribute_publication_to_profile=True,
)


# build nanopubs for many flora entries
- start with a small batch on the test server
- rows with an unmappable outcome or missing required fields are skipped and reported
- duplicate rows (same work, original and outcome) are dropped so the same claim isn't published twice
- rows in `multi_study_ids` get work = `url_r` and collection = `doi_r` / `alt_identifier_r`

In [ ]:
batch = test   

# drop duplicate claims (same replication, original and outcome).
# url_r and alt_identifier_r are included so multi-study rows sharing a doi_r aren't dropped.
dup_cols = ["doi_r", "alt_identifier_r", "url_r", "doi_o", "outcome"]
n_dups = batch.duplicated(dup_cols).sum()
batch = batch.drop_duplicates(dup_cols)
print(f"Dropped {n_dups} duplicate rows")

nanopubs_built = []   # (flora id, Nanopub) pairs
skipped = []          # (flora id, reason) pairs

for _, row in batch.iterrows():
    try:
        nanopubs_built.append((row["id"], build_nanopub(row, np_conf, multi_study_ids)))
    except ValueError as e:
        skipped.append((row["id"], str(e)))

n_multi = sum(i in multi_study_ids for i, _ in nanopubs_built)
print(f"Built {len(nanopubs_built)} nanopubs ({n_multi} multi-study), skipped {len(skipped)} rows")
skipped_df = pd.DataFrame(skipped, columns=["id", "reason"])
skipped_df.reason.value_counts()

print(nanopubs_built[0][1].provenance.serialize(format="turtle"))


Dropped 0 duplicate rows
Built 2 nanopubs (1 multi-study), skipped 0 rows


Series([], Name: count, dtype: int64)

In [83]:
# inspect a few before publishing
for flora_id, np_ in nanopubs_built:
    print(flora_id)
    print(np_.assertion.serialize(format="turtle"))

FLORA-000162
@prefix cito: <http://purl.org/spar/cito/> .
@prefix forrt: <https://w3id.org/sciencelive/o/terms/> .

<https://doi.org/10.1097/mlr.0000000000002256> a forrt:FORRT-Replication-Study ;
    cito:confirms <https://doi.org/10.1007/s11606-021-06676-x> ;
    forrt:FORRT-Replication-Outcome "In the matched samples, CCP patients had a significantly lower hazard of 180-day mortality compared with UPC in this replication cohort (0.71, 95% CI: 0.61 61 82)." .


FLORA-002642
@prefix cito: <http://purl.org/spar/cito/> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix forrt: <https://w3id.org/sciencelive/o/terms/> .

<https://osf.io/4rch2> a forrt:FORRT-Replication-Study ;
    dcterms:isPartOf <https://doi.org/10.1038/s41586-025-10078-y> ;
    cito:confirms <https://doi.org/10.1016/j.socscimed.2017.03.035> ;
    forrt:FORRT-Replication-Outcome "This claim was replicated. The OLS slope coefficient on ‘three or more siblings’ isnegative (-5.14557), and has p-value of 0.000028 (two-

# publish (not run yet)
- every published nanopub is logged to a CSV (separate files for test and real server)
- on a rerun, flora ids already in the log are skipped, so nothing is published twice
- short pause between publishes so the server isn't flooded

In [84]:
import csv, os, time
from datetime import datetime, timezone
from nanopub.definitions import TEST_NANOPUB_REGISTRY_URL

# the library swaps in the test server internally, so work out the real target here
SERVER = TEST_NANOPUB_REGISTRY_URL if np_conf.use_test_server else np_conf.use_server
LOG_FILE = "published_test.csv" if np_conf.use_test_server else "published.csv"
LOG_COLS = ["id", "nanopub", "server", "published_at"]

def already_published():
    if not os.path.exists(LOG_FILE):
        return set()
    return set(pd.read_csv(LOG_FILE)["id"])

def publish_batch(built, pause=0.5):
    done = already_published()
    todo = [(i, n) for i, n in built if i not in done]
    print(f"{len(built) - len(todo)} already published, publishing {len(todo)} to {SERVER}")

    new_file = not os.path.exists(LOG_FILE)
    with open(LOG_FILE, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_COLS)
        if new_file:
            writer.writeheader()
        for flora_id, np_ in todo:
            try:
                np_.publish()
            except Exception as e:
                print(f"{flora_id}: failed - {e}")
                continue
            writer.writerow({
                "id": flora_id,
                "nanopub": np_.source_uri,
                "server": SERVER,
                "published_at": datetime.now(timezone.utc).isoformat(),
            })
            f.flush()   # write each row immediately, so a crash doesn't lose the log
            print(f"{flora_id}: {np_.source_uri}")
            time.sleep(pause)

# uncomment when ready:
publish_batch(nanopubs_built)

0 already published, publishing 2 to https://test.registry.knowledgepixels.com/np/
FLORA-000162: https://w3id.org/np/RAMMXigN5sLuAvaKrboL6DeRh5zs_OEMXQggZM4oXhkgI
FLORA-002642: https://w3id.org/np/RA0R2KHp8Fbgs8zydR-0o2EFS_alZhJeqloeSLCLc1gfo


# How to retract if necessary

In [ ]:
# import csv, os, time
# from datetime import datetime, timezone
# from nanopub import NanopubConf, load_profile
# from nanopub.templates.nanopub_retract import NanopubRetract

# # real network, NOT the test server
# real_conf = NanopubConf(
#     profile=load_profile(),
#     use_test_server=False,
# )

# RETRACT_LOG = "retracted.csv"

# published = pd.read_csv("published.csv")
# done = set(pd.read_csv(RETRACT_LOG)["nanopub"]) if os.path.exists(RETRACT_LOG) else set()
# to_retract = [u for u in published["nanopub"] if u not in done]
# print(f"{len(done)} already retracted, {len(to_retract)} to retract on {real_conf.use_server}")

# # build and inspect first
# retractions = [(uri, NanopubRetract(conf=real_conf, uri=uri)) for uri in to_retract]
# print(retractions[0][1].assertion.serialize(format="turtle"))


0 already retracted, 2 to retract on https://registry.petapico.org/np/
@prefix npx: <http://purl.org/nanopub/x/> .
@prefix orcid: <https://orcid.org/> .

orcid:0000-0001-9911-526X npx:retracts <https://w3id.org/np/RA373Yb0rpuiK3md7pwK0Z25nh5LF0IfMe4D2VpWKKCT0> .


